# L07 -- Physics-informed Neural Network

Companion notebook for L07. Builds the toy $(V, \ell)$ classifier from the slides, trains a plain MLP, then a PI-MLP with the limit-aware penalty.

**Prerequisite**: L05. Slides: `L07_pinn.pdf`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
print(f"torch {torch.__version__}")

## 1. Toy data

Ground-truth rule: unsafe iff $V < 0.95$ or $\ell > 1.0$. 30 labeled points sampled near the boundary.

In [ ]:
def true_label(V, I):
    return ((V < 0.95) | (I > 1.0)).float()

# 30 training points clustered near the corner of the safe rectangle
torch.manual_seed(0)
n = 30
V_tr = 0.92 + 0.10 * torch.rand(n)
I_tr = 0.85 + 0.25 * torch.rand(n)
X_tr = torch.stack([V_tr, I_tr], dim=1)
y_tr = true_label(V_tr, I_tr).unsqueeze(1)

# Physics points: uniform across the whole feature space, no labels needed for plain MLP
V_phys = 0.85 + 0.25 * torch.rand(1000)
I_phys = 0.60 + 0.70 * torch.rand(1000)
X_phys = torch.stack([V_phys, I_phys], dim=1)
print(f"Training: {n} labeled points. Physics: {len(X_phys)} unlabeled points.")

## 2. Plain MLP -- baseline

In [ ]:
def make_mlp():
    return nn.Sequential(
        nn.Linear(2, 32), nn.ReLU(),
        nn.Linear(32, 32), nn.ReLU(),
        nn.Linear(32, 1), nn.Sigmoid())

torch.manual_seed(0)
plain = make_mlp()
opt = torch.optim.Adam(plain.parameters(), lr=1e-3, weight_decay=1e-4)
bce = nn.BCELoss()
for _ in range(500):
    opt.zero_grad()
    bce(plain(X_tr), y_tr).backward()
    opt.step()

## 3. PI-MLP with limit-aware physics loss

In [ ]:
def physics_loss(x, y_hat):
    V, I = x[:, 0], x[:, 1]
    should_unsafe = ((V < 0.95) | (I > 1.0)).float().unsqueeze(1)
    should_safe   = 1.0 - should_unsafe
    pen_unsafe = torch.relu(0.5 - y_hat) ** 2 * should_unsafe
    pen_safe   = torch.relu(y_hat - 0.5) ** 2 * should_safe
    return (pen_unsafe + pen_safe).mean()

torch.manual_seed(0)
pinn = make_mlp()
opt = torch.optim.Adam(pinn.parameters(), lr=1e-3, weight_decay=1e-4)
lam = 1.0
for _ in range(500):
    opt.zero_grad()
    loss_data = bce(pinn(X_tr), y_tr)
    loss_phys = physics_loss(X_phys, pinn(X_phys))
    (loss_data + lam * loss_phys).backward()
    opt.step()

## 4. Decision-boundary comparison

In [ ]:
VV, II = np.meshgrid(np.linspace(0.85, 1.10, 200),
                     np.linspace(0.60, 1.30, 200))
grid = torch.tensor(np.c_[VV.ravel(), II.ravel()], dtype=torch.float32)

with torch.no_grad():
    plain_p = plain(grid).numpy().reshape(VV.shape)
    pinn_p  = pinn(grid).numpy().reshape(VV.shape)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, p, name in zip(axes, [plain_p, pinn_p], ["Plain MLP", "PI-MLP"]):
    ax.contourf(VV, II, p, levels=20, cmap="RdBu_r", alpha=0.7)
    ax.contour(VV, II, p, levels=[0.5], colors="black", linewidths=2)
    # True boundary (L-shape)
    ax.plot([0.95, 0.95], [0.60, 1.0], "g--", linewidth=2, label="true boundary")
    ax.plot([0.95, 1.10], [1.0, 1.0],  "g--", linewidth=2)
    ax.scatter(X_tr[:, 0], X_tr[:, 1], c=y_tr.squeeze(), cmap="bwr", edgecolor="k", s=25)
    ax.set_title(name); ax.set_xlabel("$V$"); ax.set_ylabel("$\ell$"); ax.legend(loc="upper right")
plt.tight_layout(); plt.show()

## 5. Quantify the extrapolation gain

Sample uniformly from the full feature space, compute true labels, and measure accuracy of plain vs PI-MLP.

In [ ]:
torch.manual_seed(1)
V_test = 0.85 + 0.25 * torch.rand(2000)
I_test = 0.60 + 0.70 * torch.rand(2000)
X_test = torch.stack([V_test, I_test], dim=1)
y_test = true_label(V_test, I_test)

with torch.no_grad():
    acc_plain = ((plain(X_test).squeeze() >= 0.5).float() == y_test).float().mean().item()
    acc_pinn  = ((pinn(X_test).squeeze()  >= 0.5).float() == y_test).float().mean().item()

print(f"Plain MLP accuracy on uniform test set: {acc_plain:.3f}")
print(f"PI-MLP   accuracy on uniform test set: {acc_pinn:.3f}")
print("The gap is the value of the physics constraint.")

## Homework

### Required
1. Sweep $\lambda \in \{0, 0.01, 0.1, 1.0, 10, 100\}$. Plot accuracy vs $\lambda$. Where is the sweet spot?
2. Reduce the training set from 30 to 10 points. Does the PI-MLP gap widen?
3. Replace the hard rule with a soft margin: penalty proportional to $\max(0, 0.95 - V + 0.02)$. Re-train. Does the boundary become smoother?

### Optional
- Add a third feature (VUF) and a third limit (2%). Implement the limit-aware loss for three quantities.
- Replace the MLP backbone with the GCN from L06 and apply the limit penalty per node.

In [ ]:
# TODO Q1: lambda sweep
# ...

# TODO Q2: smaller training set
# ...

# TODO Q3: soft margin penalty
# ...